In [1]:
import numpy as np
import pandas as pd
import math

In [2]:
def _log(probability):
    if probability == 0:
        return -math.inf
    return math.log(probability)

In [6]:
class HiddenMarkov:
    def __init__(self, init_probs, trans_probs, emit_probs):
        self.states = tuple(init_probs)
        self.init_probs = init_probs
        self.trans_probs = trans_probs
        self.emit_probs = emit_probs


    def Viterbi(self, observations):
        if not observations:
            return  [], 0.0
        
        alphabet = set()
        
        for emissions in self.emit_probs.values():
            alphabet.update(emissions.keys())

        for symbol in observations:
            if symbol not in alphabet:
                raise ValueError(f"Invalid Observation: {symbol!r}")
        
        # create score table and previous state tables
        scores = []
        previous = []

        for symbol in observations:
            score_column = {}
            previous_column = {}

            for state in self.states:
                score_column[state] = float("-inf")
                previous_column[state] = None
            
            scores.append(score_column)
            previous.append(previous_column)

        # Initialization 
        for state in self.states:
            scores[0][state] = (
            _log(self.init_probs[state]) 
            + _log(self.emit_probs[state][observations[0]])
            )
        
        # Recursion 
        for position in range(1, len(observations)):
            for current_state in self.states:
                for preceding_state in self.states:
                    candidate = (
                        scores[position - 1][preceding_state]
                        + _log(self.trans_probs[preceding_state][current_state])
                        + _log(self.emit_probs[current_state][observations[position]])
                    )
                    if candidate > scores[position][current_state]:
                        scores[position][current_state] = candidate
                        previous[position][current_state] = preceding_state

        # Existing debugging output
        for position, symbol in enumerate(observations):
            print(f"Position {position}, observation {symbol}")

            for state in self.states:
                print(
                    f"  State {state}: "
                    f"score={scores[position][state]:.6f}, "
                    f"previous={previous[position][state]}"
                    )
        # TERMINATION
        final = max(scores[-1], key = scores[-1].get)
        best_log = scores[-1][final]
        print("Best final state:", final)
        print("Best log probability:", best_log)

        if best_log == -math.inf:
            raise ValueError("No Valid path can produce the observations.")
        
        # TRACEBACK
        path = [final]
        
        for position in range(len(observations) - 1, 0, -1):
            preceding_state = previous[position][path[-1]]
            path.append(preceding_state)

        path.reverse()

        return path, best_log

    


In [7]:
# Example observation sequence
obs = "GGCACTGAA"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.2,
    "G": 0.8
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.7, "G": 0.3},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

In [8]:
model = HiddenMarkov(init_probs, trans_probs, emit_probs)

best_path, best_log = model.Viterbi(obs)

print("Best path:", best_path)
print("Best log probability:", best_log)

Position 0, observation G
  State I: score=-2.525729, previous=None
  State G: score=-1.832581, previous=None
Position 1, observation G
  State I: score=-3.798694, previous=I
  State G: score=-3.547380, previous=G
Position 2, observation C
  State I: score=-5.071660, previous=I
  State G: score=-5.262178, previous=G
Position 3, observation A
  State I: score=-7.730920, previous=I
  State G: score=-6.571512, previous=G
Position 4, observation C
  State I: score=-9.003886, previous=I
  State G: score=-8.286310, previous=G
Position 5, observation T
  State I: score=-11.663146, previous=I
  State G: score=-9.595643, previous=G
Position 6, observation G
  State I: score=-12.814519, previous=G
  State G: score=-11.310442, previous=G
Position 7, observation A
  State I: score=-15.473779, previous=I
  State G: score=-12.619775, previous=G
Position 8, observation A
  State I: score=-17.224945, previous=G
  State G: score=-13.929108, previous=G
Best final state: G
Best log probability: -13.92910

![](assets/rotation.png)

![](assets/rotation.png)